# Prepare the biological datasets

Download the GSE52529 FPKM matrix and its GEO sample metadata, plus Scanpy’s Paul15 and PBMC3k inputs. The outputs stay under the locally ignored `data/` directory; no dataset files are committed. Run this once before [the biological comparison notebook](../biological_exp.ipynb), then rerun it only if an input is missing or damaged.

Install Scanpy in the active kernel if needed: `%pip install scanpy`. Sources: [GSE52529 on GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE52529), [Scanpy dataset API](https://scanpy.readthedocs.io/en/stable/api/datasets.html).


In [ ]:
from pathlib import Path
import gzip
import shutil
import urllib.request

import pandas as pd

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "imste" / "__init__.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the IMSTE repository from the notebook working directory.")

DATA_DIR = REPO_ROOT / "data"
GSE_DIR = DATA_DIR / "gse52529"
PAUL_DIR = DATA_DIR / "paul15"
GSE_DIR.mkdir(parents=True, exist_ok=True)
PAUL_DIR.mkdir(parents=True, exist_ok=True)

GSE_MATRIX = GSE_DIR / "GSE52529_fpkm_matrix.txt.gz"
GEO_FAMILY = GSE_DIR / "GSE52529_family.soft.gz"
GSE_METADATA = GSE_DIR / "GSE52529_cell_metadata.csv"
PAUL15_PATH = PAUL_DIR / "paul15.h5"
PBMC_RAW_PATH = DATA_DIR / "pbmc3k_raw.h5ad"
PBMC_PROCESSED_PATH = DATA_DIR / "pbmc3k_processed.h5ad"

GEO_MATRIX_URL = (
    "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE52529&format=file&file="
    "GSE52529_fpkm_matrix.txt.gz"
)
GEO_FAMILY_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE52nnn/GSE52529/soft/"
    "GSE52529_family.soft.gz"
)
FORCE_GEO_DOWNLOAD = False


def valid_gzip(path):
    try:
        with gzip.open(path, "rb") as stream:
            if not stream.read(128):
                return False
            while stream.read(8 * 1024 * 1024):
                pass
        return True
    except (OSError, EOFError):
        return False


def download_gzip(url, path, *, force=False):
    if not force and valid_gzip(path):
        print(f"Using existing {path.name}")
        return
    partial_path = path.with_name(path.name + ".partial")
    partial_path.unlink(missing_ok=True)
    request = urllib.request.Request(
        url, headers={"User-Agent": "IMSTE biological dataset preparation"}
    )
    print(f"Downloading {path.name}...")
    try:
        with urllib.request.urlopen(request, timeout=120) as response, partial_path.open("wb") as output:
            shutil.copyfileobj(response, output, length=1024 * 1024)
        if not valid_gzip(partial_path):
            raise RuntimeError(f"The downloaded file {path.name} is not a valid gzip archive.")
        partial_path.replace(path)
    finally:
        partial_path.unlink(missing_ok=True)


download_gzip(GEO_MATRIX_URL, GSE_MATRIX, force=FORCE_GEO_DOWNLOAD)
download_gzip(GEO_FAMILY_URL, GEO_FAMILY, force=FORCE_GEO_DOWNLOAD)

# GEO's SOFT family file supplies sample titles and QC characteristics.
sample_records = []
current = None
with gzip.open(GEO_FAMILY, "rt", encoding="utf-8") as soft_file:
    for line in soft_file:
        line = line.rstrip("\n")
        if line.startswith("^SAMPLE = "):
            if current is not None:
                sample_records.append(current)
            current = {"geo_accession": line.split("=", 1)[1].strip(), "characteristics": {}}
        elif current is not None and line.startswith("!Sample_title = "):
            title = line.split("=", 1)[1].strip()
            current["sample_id"] = title.removeprefix("Cell ")
        elif current is not None and line.startswith("!Sample_characteristics_ch1 = "):
            key, separator, value = line.split("=", 1)[1].strip().partition(":")
            if separator:
                current["characteristics"][key.strip().lower()] = value.strip()
if current is not None:
    sample_records.append(current)

matrix_sample_ids = list(pd.read_csv(GSE_MATRIX, sep="\t", compression="gzip", nrows=0).columns)
record_by_sample = {record.get("sample_id"): record for record in sample_records}
missing_samples = [sample_id for sample_id in matrix_sample_ids if sample_id not in record_by_sample]
if missing_samples:
    raise RuntimeError(f"GEO metadata is missing matrix samples: {missing_samples[:5]}")

metadata_rows = []
for sample_id in matrix_sample_ids:
    record = record_by_sample[sample_id]
    characteristics = record["characteristics"]
    required = {"hour post serum-switch", "debris", "control well", "cells in well"}
    absent = required - characteristics.keys()
    if absent:
        raise RuntimeError(f"GEO sample {record['geo_accession']} lacks fields: {sorted(absent)}")
    debris = characteristics["debris"].lower() == "true"
    control_well = characteristics["control well"].lower() == "true"
    cells_in_well = characteristics["cells in well"]
    metadata_rows.append({
        "sample_id": sample_id,
        "geo_accession": record["geo_accession"],
        "time_hours": int(characteristics["hour post serum-switch"]),
        "debris": debris,
        "control_well": control_well,
        "cells_in_well": cells_in_well,
        "include_clean_single_cell": not debris and not control_well and cells_in_well == "1",
    })

metadata = pd.DataFrame(metadata_rows)
metadata.to_csv(GSE_METADATA, index=False)
print(f"GSE52529: {len(matrix_sample_ids)} matrix samples; "
      f"{metadata['include_clean_single_cell'].sum()} clean single-cell samples.")


In [ ]:
import scanpy as sc
from IPython.display import display

# These Scanpy calls download/cache the files at the paths used by the comparison loader.
sc.settings.datasetdir = DATA_DIR
paul15 = sc.datasets.paul15()
pbmc3k_raw = sc.datasets.pbmc3k()
pbmc3k_processed = sc.datasets.pbmc3k_processed()

expected_paths = (PAUL15_PATH, PBMC_RAW_PATH, PBMC_PROCESSED_PATH)
missing = [path for path in expected_paths if not path.is_file()]
if missing:
    raise RuntimeError(f"Scanpy did not create expected file(s): {missing}")

prepared = pd.DataFrame([
    {"dataset": "GSE52529", "file": GSE_MATRIX.name, "size_MB": round(GSE_MATRIX.stat().st_size / 1e6, 1)},
    {"dataset": "GSE52529 metadata", "file": GSE_METADATA.name, "size_MB": round(GSE_METADATA.stat().st_size / 1e6, 1)},
    {"dataset": "Paul15", "cells": paul15.n_obs, "file": str(PAUL15_PATH.relative_to(DATA_DIR)), "size_MB": round(PAUL15_PATH.stat().st_size / 1e6, 1)},
    {"dataset": "PBMC3k raw", "file": PBMC_RAW_PATH.name, "size_MB": round(PBMC_RAW_PATH.stat().st_size / 1e6, 1)},
    {"dataset": "PBMC3k processed", "file": PBMC_PROCESSED_PATH.name, "size_MB": round(PBMC_PROCESSED_PATH.stat().st_size / 1e6, 1)},
])
display(prepared)
